In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_gold/10_dim_medication
# Purpose  : SCD Type 1 for medication codes (178 distinct codes based on profiling).
#            Follows the exact same pattern as dim_condition, dim_procedure, and dim_observation_type.
#            There is no risk analogous to the units issue in dim_observation_type here—
#            medication_code and description are practically static, while transactional
#            attributes (DISPENSES, BASE_COST, etc.) remain in the Fact table.
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.view(name="v_medication_type_changes")
def v_medication_type_changes():
    df = dlt.read_stream("covid19_lakehouse.silver.medications")
    return df.select(
        F.xxhash64("medication_code").alias("medication_key"),
        "medication_code",
        "medication_description",
        "_ingested_at",
    )


dlt.create_streaming_table("covid19_lakehouse.gold.dim_medication")

dlt.apply_changes(
    target="covid19_lakehouse.gold.dim_medication",
    source="v_medication_type_changes",
    keys=["medication_code"],
    sequence_by=F.col("_ingested_at"),
    stored_as_scd_type=1,
)